In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No CUDA')

2.11.0+cu126
12.6
True
Tesla T4


In [ ]:
import os, re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from pathlib import Path
import glob
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from huggingface_hub import login

## Load tokenizer and model

In [ ]:
HF_TOKEN = os.getenv("HF_TOKEN")
login(token=HF_TOKEN)

In [ ]:
MODELS = ["Qwen/Qwen3-0.6B", "Qwen/Qwen3-1.7B", "Qwen/Qwen3-4B", "Qwen/Qwen3-8B"]

def load(MODEL_NAME):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )
    model.eval()
    model = torch.compile(model, mode="reduce-overhead")
    print(f"Model loaded on : {next(model.parameters()).device}")
    return tokenizer, model

## ELGOLD Dataset

In [ ]:
DATA_DIR     = "/content/data"
BATCH_SIZE = 8
MAX_EXAMPLES = 200

In [ ]:
ANNOT_RE = re.compile(r"\{\{([^|{}]+)\|([^|{}]+)\|([^{}]+)\}\}")

def clean_text_and_extract_mentions(path, context_window=150):
    raw = Path(path).read_text(encoding="utf-8")
    examples = []

    # All annotations replaced
    clean_text = ANNOT_RE.sub(lambda m: m.group(1).strip(), raw)

    # Extracting context
    cursor_raw = 0
    cursor_clean = 0
    last_end_raw = 0

    for match in ANNOT_RE.finditer(raw):
        mention = match.group(1).strip()
        ent_type = match.group(2).strip()
        target = match.group(3).strip()

        # Characters between previous match end and this match start
        literal_before = raw[last_end_raw:match.start()]
        cursor_clean += len(literal_before)

        mention_start = cursor_clean
        mention_end = cursor_clean + len(mention)

        # Extracting a symmetric character window around the mention
        ctx_start = max(0, mention_start - context_window)
        ctx_end = min(len(clean_text), mention_end + context_window)
        context_snippet = clean_text[ctx_start:ctx_end]

        if ctx_start > 0:
            context_snippet = "…" + context_snippet
        if ctx_end < len(clean_text):
            context_snippet = context_snippet + "…"

        examples.append({
            "file": Path(path).name,
            "mention": mention,
            "entity_type": ent_type,
            "gold": target,
            "context": context_snippet,
        })

        # Advancing cursors
        cursor_clean += len(mention)
        last_end_raw = match.end()

    return examples

rows = []
for file in glob.glob(f"{DATA_DIR}/*.txt"):
    rows.extend(clean_text_and_extract_mentions(file))

df_gold = pd.DataFrame(rows).drop_duplicates()
if MAX_EXAMPLES:
    df_gold = df_gold.head(MAX_EXAMPLES)

print(df_gold.head())
print("Examples number:", len(df_gold))

      file                       mention entity_type  \
0  2_3.txt           Toronto Film School         ORG   
1  2_3.txt  QS World University Rankings         ORG   
4  2_3.txt                 First Nations        NORP   
5  2_3.txt                         Métis        NORP   
6  2_3.txt                         Inuit        NORP   

                           gold  \
0           Toronto Film School   
1  QS World University Rankings   
4       First Nations in Canada   
5                         Métis   
6                         Inuit   

                                             context  
0  Toronto Film School, ranked as a top film scho...  
1  Toronto Film School, ranked as a top film scho...  
4  Toronto Film School, ranked as a top film scho...  
5  Toronto Film School, ranked as a top film scho...  
6  Toronto Film School, ranked as a top film scho...  
Nombre d'exemples: 2130


## Prompt zero-shot

In [ ]:
def build_prompt(text: str, mention: str, entity_type: str, model_name: str) -> str:
    prompt = """/no_think
You are a precise English Wikipedia Entity Linking system.

Your task is to map one entity mention in context to its canonical English Wikipedia article title.

Input:
TEXT: {text}
MENTION: {mention}
ENTITY_TYPE: {entity_type}

Instructions:
- Identify the real-world entity referred to by MENTION in TEXT.
- Use TEXT as the main evidence.
- Use ENTITY_TYPE to avoid linking to the wrong kind of entity.
- Resolve abbreviations, acronyms, aliases, nicknames, surnames, demonyms, adjectival forms, shortened names, and ambiguous mentions.
- Return the canonical English Wikipedia article title.
- Use spaces, not underscores.
- Preserve Wikipedia capitalization.
- Include parenthetical disambiguation only when it is part of the Wikipedia article title.
- If the mention is already the correct title, return it unchanged.
- If uncertain, return the single most plausible English Wikipedia article title supported by TEXT.

Output constraints:
- Output only one title.
- Do not explain.
- Do not output JSON.
- Do not output Markdown.
- Do not add quotes.
- Do not add punctuation after the title.
- Do not output labels.
- Do not output braces.
- Do not output template variables.

Answer:"""

    return prompt

In [ ]:
def generate_batch(prompts: list[str], tokenizer, model, max_new_tokens=20) -> list[str]:
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    results = []
    input_len = inputs["input_ids"].shape[1]
    for out in outputs:
        new_tokens = out[input_len:]
        results.append(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())
    return results

In [ ]:
def parse_llm_response(raw: str) -> str:
    for line in raw.splitlines():
        line = line.strip()
        if line:
            return line
    return raw.strip()

In [ ]:
# Test sur le 1er exemple
row = df_gold.iloc[0]
print("PROMPT :")
print("-" * 60)
print(build_prompt(row["context"], row["mention"], row["entity_type"]))
print("-" * 60)
print(f"Gold attendu : {row['gold']}")

PROMPT :
------------------------------------------------------------
You are an Entity Linking system. Your task is to identify the entity that a mention refers to in a text, and return its canonical name as it appears in a Knowledge Base (e.g. Wikidata / Wikipedia).

Text: "Toronto Film School, ranked as a top film school by the QS World University Rankings, offers a fast-paced, hands-on and highly collaborative learning environment, designed to arm you with the theory and skills needed to align your career aspirations with your creative talents. Whether you are passionate about film, design, video games or fashion, Toronto Film School is where you will find a supportive community, award-winning instructors, and a network of creative peers to propel you into the career you dream of.
At Toronto Film School, we are committed to making Diversity, Equity and Inclusion an integral part of our learning and working environments.
We achieve this by:
Celebrating the full range of human divers

## Zero-shot Evaluation

In [ ]:
def evaluate_model(tokenizer, model, model_name, df_gold, batch_size=8, match_fn=None):
    match_fn = match_fn or (lambda p, g: p.lower().strip() == g.lower().strip())
    results = []

    rows_data = [
        {**row, "prompt": build_prompt(row["context"], row["mention"], row["entity_type"])}
        for row in df_gold.to_dict("records")
    ]

    for i in tqdm(range(0, len(rows_data), batch_size), desc=f"Zero-shot {model_name}"):
        batch = rows_data[i:i + batch_size]
        batch_prompts = [item["prompt"] for item in batch]

        try:
            batch_outputs = generate_batch(batch_prompts, tokenizer, model)
        except Exception as e:
            print(f"[WARNING] Batch {i} failed ({type(e).__name__}: {e}) — skipping {len(batch)} items")
            for item in batch:
                results.append({
                    "file": item["file"],
                    "mention": item["mention"],
                    "entity_type": item["entity_type"],
                    "gold": item["gold"],
                    "predicted": "",
                    "correct": False,
                    "raw_output": "",
                    "error": str(e),
                })
            continue

        for item, raw_output in zip(batch, batch_outputs):
            try:
                pred = parse_llm_response(raw_output)
                error = ""
            except Exception as e:
                print(f"[WARNING] parse_llm_response failed for '{item['mention']}': {e}")
                pred = ""
                error = str(e)

            results.append({
                "file": item["file"],
                "mention": item["mention"],
                "entity_type": item["entity_type"],
                "gold": item["gold"],
                "predicted": pred,
                "correct": False if error else match_fn(pred, item["gold"]),
                "raw_output": raw_output,
                "error": error,
            })

    return pd.DataFrame(results)

In [ ]:
results = dict()

for MODEL in MODELS:
    print(f"\n{'='*50}\nModel : {MODEL}\n{'='*50}")

    tokenizer, model = load(MODEL)
    results[MODEL] = evaluate_model(tokenizer, model, MODEL, df_gold, BATCH_SIZE)

    df_results = results[MODEL][results[MODEL]["error"] == ""]
    total = len(df_results)
    acc = df_results["correct"].mean()

    print("Zero-shot ELGOLD results:\n")
    print(f"Model: {MODEL}")
    print(f"Tested examples: {total}")
    print(f"Accuracy: {acc:.2%}  ({df_results['correct'].sum()}/{total})")

    model_slug = MODEL.replace("/", "_")
    results[MODEL].to_csv(f"zero_shot_elgold_{model_slug}.csv", index=False)

    wrong = df_results[~df_results["correct"]]
    print(f"\n{len(wrong)} errors :")
    for _, r in wrong.head(10).iterrows():
        print(f"  '{r['mention']}' | gold={r['gold']} | pred={r['predicted']}")

    # Free GPU memory before the next model
    del model, tokenizer
    torch.cuda.empty_cache()


Model : Qwen/Qwen3-1.7B


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Modèle chargé sur : cuda:0


Zero-shot Qwen/Qwen3-1.7B:  78%|███████▊  | 105/134 [31:32<08:52, 18.37s/it]

In [ ]:
results["Qwen/Qwen3-0.6B"][:10]

Model: Qwen/Qwen3-0.6B

Tested examples: 2130

Accuracy: 1.97%  (42/2130)

- **Model: Qwen/Qwen3-0.6B**

Tested examples: 2130

Accuracy: 25.26%  (538/2130)

- **Model: Qwen/Qwen3-1.7B**

Tested examples: 200

Accuracy: 40.50%  (81/200)

- **Model: Qwen/Qwen3-4B**

Tested examples: 200

Accuracy: 44.50%  (89/200)

- **Model: Qwen/Qwen3-8B**

Tested examples: 200

Accuracy: 56.00%  (112/200)

## Model comparison

In [ ]:
rows = []
for MODEL, res in results.items():
    df_r = pd.DataFrame(res)
    rows.append({
        "model":     MODEL,
        "accuracy":  df_r["correct"].mean(),
        "n_correct": int(df_r["correct"].sum()),
        "n_total":   len(df_r),
    })

df_compare = pd.DataFrame(rows).sort_values("accuracy", ascending=False)
df_compare["accuracy"] = df_compare["accuracy"].map("{:.2%}".format)
print("ZERO-SHOT Model comparison")
print(df_compare.to_string(index=False))